### HF format → ESM format conversion (lm_head excluded)

In [3]:
import os
import re
import glob
import torch
from safetensors.torch import load_file

# Specify directory to search for .safetensors (converted .pt is also saved here)
INPUT_DIR = "./models/sft_esm2_8m_head_only_optuna/encoder"  # Change as needed

pattern = os.path.join(INPUT_DIR, "*.safetensors")
candidates = glob.glob(pattern)

# Prefer model.safetensors; otherwise use the first .safetensors found
input_path = None
for p in candidates:
    if os.path.basename(p) == "model.safetensors":
        input_path = p
        break
if input_path is None and candidates:
    input_path = candidates[0]

if input_path is None:
    raise FileNotFoundError(f"No .safetensors file found in directory: {INPUT_DIR}")

out_path = os.path.join(INPUT_DIR, "converted_model.pt")

# Map HF keys to ESM keys (aligned with verification and downstream)
LAYER_RE = re.compile(r"^encoder\.layer\.(\d+)\.")

def map_hf_key_to_esm(hf_key: str):
    """Map HF (.safetensors) key to ESM (.pt) key. Returns (esm_key, keep)."""
    m = LAYER_RE.match(hf_key)
    layer = int(m.group(1)) if m else None

    if hf_key == "embeddings.word_embeddings.weight":
        return "esm.embed_tokens.weight", True
    if hf_key == "embeddings.position_embeddings.weight":
        return None, False  # exclude (rotary)

    if hf_key == "encoder.emb_layer_norm_after.weight":
        return "esm.emb_layer_norm_after.weight", True
    if hf_key == "encoder.emb_layer_norm_after.bias":
        return "esm.emb_layer_norm_after.bias", True

    if hf_key == "contact_head.regression.weight":
        return "esm.contact_head.regression.weight", True
    if hf_key == "contact_head.regression.bias":
        return "esm.contact_head.regression.bias", True

    if hf_key == "pooler.dense.weight":
        return "head.0.weight", True
    if hf_key == "pooler.dense.bias":
        return "head.0.bias", True

    if layer is not None:
        if hf_key.endswith(".attention.self.rotary_embeddings.inv_freq"):
            return f"esm.layers.{layer}.self_attn.rot_emb.inv_freq", True
        if hf_key.endswith(".attention.self.query.weight"):
            return f"esm.layers.{layer}.self_attn.q_proj.weight", True
        if hf_key.endswith(".attention.self.query.bias"):
            return f"esm.layers.{layer}.self_attn.q_proj.bias", True
        if hf_key.endswith(".attention.self.key.weight"):
            return f"esm.layers.{layer}.self_attn.k_proj.weight", True
        if hf_key.endswith(".attention.self.key.bias"):
            return f"esm.layers.{layer}.self_attn.k_proj.bias", True
        if hf_key.endswith(".attention.self.value.weight"):
            return f"esm.layers.{layer}.self_attn.v_proj.weight", True
        if hf_key.endswith(".attention.self.value.bias"):
            return f"esm.layers.{layer}.self_attn.v_proj.bias", True
        if hf_key.endswith(".attention.output.dense.weight"):
            return f"esm.layers.{layer}.self_attn.out_proj.weight", True
        if hf_key.endswith(".attention.output.dense.bias"):
            return f"esm.layers.{layer}.self_attn.out_proj.bias", True
        if hf_key.endswith(".attention.LayerNorm.weight"):
            return f"esm.layers.{layer}.self_attn_layer_norm.weight", True
        if hf_key.endswith(".attention.LayerNorm.bias"):
            return f"esm.layers.{layer}.self_attn_layer_norm.bias", True
        if hf_key.endswith(".intermediate.dense.weight"):
            return f"esm.layers.{layer}.fc1.weight", True
        if hf_key.endswith(".intermediate.dense.bias"):
            return f"esm.layers.{layer}.fc1.bias", True
        if hf_key.endswith(".output.dense.weight"):
            return f"esm.layers.{layer}.fc2.weight", True
        if hf_key.endswith(".output.dense.bias"):
            return f"esm.layers.{layer}.fc2.bias", True
        if hf_key.endswith(".LayerNorm.weight"):
            return f"esm.layers.{layer}.final_layer_norm.weight", True
        if hf_key.endswith(".LayerNorm.bias"):
            return f"esm.layers.{layer}.final_layer_norm.bias", True

    return None, False

print(f"Input: {input_path}")
print(f"Output: {out_path}")

hf_sd = load_file(input_path)
out_sd = {}
dropped = []
for k, v in hf_sd.items():
    out_key, keep = map_hf_key_to_esm(k)
    if not keep:
        dropped.append(k)
        continue
    if out_key is not None:
        out_sd[out_key] = v.clone()

if dropped:
    print(f"Dropped keys (no ESM counterpart): {len(dropped)} e.g. {dropped[:2]}")

torch.save(out_sd, out_path)
print(f"Done. HF keys: {len(hf_sd)} -> ESM keys: {len(out_sd)}")

Input: ./models/sft_esm2_8m_head_only_optuna/encoder/model.safetensors
Output: ./models/sft_esm2_8m_head_only_optuna/encoder/converted_model.pt
Dropped keys (no ESM counterpart): 1 e.g. ['embeddings.position_embeddings.weight']
Done. HF keys: 110 -> ESM keys: 109


### Verification code

In [4]:
import os
import re
import sys
import torch
from transformers import AutoModel

# ====== Configuration ======
HF_DIR = "./models/sft_esm2_8m_head_only_optuna/encoder"
CONVERTED_PT = "./models/sft_esm2_8m_head_only_optuna/encoder/converted_model.pt"  # Output of conversion script
CONVERTED_HAS_ESM_PREFIX = True    # True if conversion output has 'esm.' prefix (recommended)
ATOL = 0.0                         # Check exact match. Optionally allow tiny tolerance (e.g. 1e-7)
RTOL = 0.0

# ====== Deterministic HF→ESM mapping (per provided key list) ======
LAYER_RE = re.compile(r"^encoder\.layer\.(\d+)\.")

def map_hf_key_to_esm(hf_key: str):
    """
    Map HF (.safetensors) key to ESM (.pt) key.
    Returns: (esm_key, keep_flag)
      - esm_key: key in output state_dict (with 'esm.' prefix to match converted .pt)
      - keep_flag: when False, skip comparison (e.g. position_embeddings)
    """
    m = LAYER_RE.match(hf_key)
    layer = int(m.group(1)) if m else None

    # Embeddings
    if hf_key == "embeddings.word_embeddings.weight":
        out = "embed_tokens.weight"
    elif hf_key == "embeddings.position_embeddings.weight":
        return None, False  # ESM uses rotary; discard position embeddings

    # emb_layer_norm_after
    elif hf_key == "encoder.emb_layer_norm_after.weight":
        out = "emb_layer_norm_after.weight"
    elif hf_key == "encoder.emb_layer_norm_after.bias":
        out = "emb_layer_norm_after.bias"

    # contact head (also in HF)
    elif hf_key == "contact_head.regression.weight":
        out = "contact_head.regression.weight"
    elif hf_key == "contact_head.regression.bias":
        out = "contact_head.regression.bias"

    # pooler -> head.0.* (per SFT_hot.pt)
    elif hf_key == "pooler.dense.weight":
        out_full = "head.0.weight"   # no 'esm.' prefix for head in SFT_hot.pt
        return (out_full if not CONVERTED_HAS_ESM_PREFIX else out_full), True
    elif hf_key == "pooler.dense.bias":
        out_full = "head.0.bias"
        return (out_full if not CONVERTED_HAS_ESM_PREFIX else out_full), True

    # Per layer
    elif layer is not None:
        if hf_key.endswith(".attention.self.rotary_embeddings.inv_freq"):
            out = f"layers.{layer}.self_attn.rot_emb.inv_freq"
        elif hf_key.endswith(".attention.self.query.weight"):
            out = f"layers.{layer}.self_attn.q_proj.weight"
        elif hf_key.endswith(".attention.self.query.bias"):
            out = f"layers.{layer}.self_attn.q_proj.bias"
        elif hf_key.endswith(".attention.self.key.weight"):
            out = f"layers.{layer}.self_attn.k_proj.weight"
        elif hf_key.endswith(".attention.self.key.bias"):
            out = f"layers.{layer}.self_attn.k_proj.bias"
        elif hf_key.endswith(".attention.self.value.weight"):
            out = f"layers.{layer}.self_attn.v_proj.weight"
        elif hf_key.endswith(".attention.self.value.bias"):
            out = f"layers.{layer}.self_attn.v_proj.bias"
        elif hf_key.endswith(".attention.output.dense.weight"):
            out = f"layers.{layer}.self_attn.out_proj.weight"
        elif hf_key.endswith(".attention.output.dense.bias"):
            out = f"layers.{layer}.self_attn.out_proj.bias"
        elif hf_key.endswith(".attention.LayerNorm.weight"):
            out = f"layers.{layer}.self_attn_layer_norm.weight"
        elif hf_key.endswith(".attention.LayerNorm.bias"):
            out = f"layers.{layer}.self_attn_layer_norm.bias"
        elif hf_key.endswith(".intermediate.dense.weight"):
            out = f"layers.{layer}.fc1.weight"
        elif hf_key.endswith(".intermediate.dense.bias"):
            out = f"layers.{layer}.fc1.bias"
        elif hf_key.endswith(".output.dense.weight"):
            out = f"layers.{layer}.fc2.weight"
        elif hf_key.endswith(".output.dense.bias"):
            out = f"layers.{layer}.fc2.bias"
        elif hf_key.endswith(".LayerNorm.weight"):
            out = f"layers.{layer}.final_layer_norm.weight"
        elif hf_key.endswith(".LayerNorm.bias"):
            out = f"layers.{layer}.final_layer_norm.bias"
        else:
            return None, False
    else:
        return None, False

    # Add 'esm.' prefix (do not add for head.0.*)
    if out.startswith("layers.") or out.startswith("embed_") or out.startswith("emb_layer_norm_after") \
       or out.startswith("contact_head."):
        out_full = ("esm." + out) if CONVERTED_HAS_ESM_PREFIX else out
    else:
        out_full = out  # head.0.* as-is

    return out_full, True


# ====== Verification main ======
print("\n" + "="*50)
print("🔍 Starting strict verification of converted weights (lm_head.* not allowed)...")
print("="*50)

try:
    # --- Load HF ---
    hf_model = AutoModel.from_pretrained(HF_DIR, trust_remote_code=True)
    hf_sd = {k: v.cpu() for k, v in hf_model.state_dict().items()}

    # --- Load converted PT ---
    conv_sd = torch.load(CONVERTED_PT, map_location="cpu")
    conv_keys = set(conv_sd.keys())
    print(f"HF params: {len(hf_sd)}  |  Converted PT params: {len(conv_sd)}")

    # ★ Strict check: converted .pt must not contain lm_head.*
    lm_head_in_conv = sorted([k for k in conv_keys if k.startswith("esm.lm_head.") or k.startswith("lm_head.")])
    if lm_head_in_conv:
        print("❌ Error: Converted .pt contains 'lm_head.*' keys. Not allowed as they would cause overwrite.")
        for k in lm_head_in_conv[:20]:
            print("  -", k)
        sys.exit(1)

    # --- Compare values 1:1 based on HF→ESM mapping ---
    missing_in_converted, shape_mismatch, value_mismatch = [], [], []
    mapped_count = 0

    for hf_k, hf_v in hf_sd.items():
        esm_k, keep = map_hf_key_to_esm(hf_k)
        if not keep:
            continue
        mapped_count += 1

        if esm_k not in conv_sd:
            missing_in_converted.append((hf_k, esm_k))
            continue

        cv = conv_sd[esm_k].cpu()
        if hf_v.shape != cv.shape:
            shape_mismatch.append((hf_k, esm_k, tuple(hf_v.shape), tuple(cv.shape)))
            continue

        # Exact match (or allclose if needed)
        if ATOL == 0.0 and RTOL == 0.0:
            equal = torch.equal(hf_v, cv)
        else:
            equal = torch.allclose(hf_v, cv, atol=ATOL, rtol=RTOL)

        if not equal:
            max_abs = (hf_v - cv).abs().max().item()
            value_mismatch.append((hf_k, esm_k, max_abs))

    # --- List extra keys (in converted PT but not from HF) ---
    expected_from_hf = set()
    for hf_k in hf_sd.keys():
        esm_k, keep = map_hf_key_to_esm(hf_k)
        if keep and esm_k is not None:
            expected_from_hf.add(esm_k)

    extras = []
    for ck in conv_keys - expected_from_hf:
        # Allow rotary fill (from ESM reference)
        if ck.endswith(".rot_emb.inv_freq"):
            continue
        # pooler-derived head.0.* is allowed (HF pooler mapping)
        if ck.startswith("head.0."):
            continue
        # ★ lm_head.* not allowed (if we get here, NG)
        if ck.startswith("esm.lm_head.") or ck.startswith("lm_head."):
            extras.append(ck)
            continue
        # Other extras: report (normally should not appear)
        extras.append(ck)

    # --- Results ---
    print("\n--- Verification summary ---")
    print(f"Mapped (HF -> ESM keys): {mapped_count}")
    print(f"Missing in converted    : {len(missing_in_converted)}")
    print(f"Shape mismatch          : {len(shape_mismatch)}")
    print(f"Value mismatch          : {len(value_mismatch)}")
    print(f"Extras in converted     : {len(extras)}")

    if missing_in_converted:
        print("\n[Missing examples] HF -> ESM")
        for i, (h, e) in enumerate(missing_in_converted[:10], 1):
            print(f"  {i:02d}. {h}  ->  {e}")

    if shape_mismatch:
        print("\n[Shape mismatch examples] HF -> ESM (HF_shape / PT_shape)")
        for i, (h, e, s1, s2) in enumerate(shape_mismatch[:10], 1):
            print(f"  {i:02d}. {h} -> {e}  {s1} / {s2}")

    if value_mismatch:
        print("\n[Value mismatch examples] max absolute error")
        for i, (h, e, d) in enumerate(value_mismatch[:10], 1):
            print(f"  {i:02d}. {h} -> {e}  max|Δ|={d:.3e}")

    if extras:
        print("\n[Extras (in PT but not from HF)]")
        for i, k in enumerate(extras[:20], 1):
            print(f"  {i:02d}. {k}")

    # --- Final verdict ---
    print("\n--- Verdict ---")
    if (not missing_in_converted) and (not shape_mismatch) and (not value_mismatch) and (not extras):
        print("🎉 Success: Converted .pt correctly reflects HF weights and does not contain extra lm_head.*.")
    else:
        print("⚠️ Warning: There are differences. Review the details above and fix mapping/conversion.")

except Exception as e:
    print(f"Unexpected error during verification: {e}")


🔍 Starting strict verification of converted weights (lm_head.* not allowed)...
HF params: 110  |  Converted PT params: 109

--- Verification summary ---
Mapped (HF -> ESM keys): 109
Missing in converted    : 0
Shape mismatch          : 0
Value mismatch          : 0
Extras in converted     : 0

--- Verdict ---
🎉 Success: Converted .pt correctly reflects HF weights and does not contain extra lm_head.*.
